In [3]:
import os
import glob
import pandas as pd
import shutil

# find all directories matching the pattern
dir_pattern = "Decon-COO-Results_Degradation_v4_top_*_percent_removed_nuSVR/"
directories = sorted(glob.glob(dir_pattern))

print(f"Found {len(directories)} directories.")

for input_dir in directories:
    print(f"\nProcessing directory: {input_dir}")

    pattern = os.path.join(input_dir, "nuSVR_Rerun_*.txt")
    files = glob.glob(pattern)

    if not files:
        print("  No rerun files found — skipping.")
        continue

    samples = {}

    for f in files:
        name = os.path.basename(f)
        parts = name.replace(".txt", "").split("_")

        # sample ID = everything between 3rd token and C/nu
        sample_id = "_".join(parts[3:-2])

        # C and nu always at the end
        C_val = parts[-2].replace("C", "")
        nu_val = parts[-1].replace("nu", "")

        df = pd.read_csv(f, sep="\t")
        rmse = float(df["RMSE-PredictedCounts"].iloc[0])

        # keep best RMSE per sample
        if sample_id not in samples or rmse < samples[sample_id]["rmse"]:
            samples[sample_id] = {
                "rmse": rmse,
                "file": f,
                "C": C_val,
                "nu": nu_val
            }

    # copy and rename winning files
    for sample_id, info in samples.items():
        best_file = info["file"]
        rmse = info["rmse"]
        C_val = info["C"]
        nu_val = info["nu"]

        # output format you requested
        new_name = f"nuSVR_CountsRMSE_Random_{sample_id}.txt"
        new_path = os.path.join(input_dir, new_name)

        shutil.copy(best_file, new_path)

        print(f"  Sample: {sample_id}")
        print(f"    → Best RMSE: {rmse:.6f}")
        print(f"    → Best C:  {C_val}")
        print(f"    → Best nu: {nu_val}")
        print(f"    → Saved: {new_path}")


Found 4 directories.

Processing directory: Decon-COO-Results_Degradation_v4_top_10_percent_removed_nuSVR/


  Sample: TSP27_random_rep4_seed6_20_400
    → Best RMSE: 0.429608
    → Best C:  0.75
    → Best nu: 0.1
    → Saved: Decon-COO-Results_Degradation_v4_top_10_percent_removed_nuSVR/nuSVR_CountsRMSE_Random_TSP27_random_rep4_seed6_20_400.txt
  Sample: TSP25_random_rep4_seed6_20_400
    → Best RMSE: 0.422794
    → Best C:  1
    → Best nu: 0.75
    → Saved: Decon-COO-Results_Degradation_v4_top_10_percent_removed_nuSVR/nuSVR_CountsRMSE_Random_TSP25_random_rep4_seed6_20_400.txt
  Sample: TSP25_random_rep5_seed5_10_200
    → Best RMSE: 0.363001
    → Best C:  0.75
    → Best nu: 0.75
    → Saved: Decon-COO-Results_Degradation_v4_top_10_percent_removed_nuSVR/nuSVR_CountsRMSE_Random_TSP25_random_rep5_seed5_10_200.txt
  Sample: TSP27_random_rep4_seed9_50_1000
    → Best RMSE: 0.359113
    → Best C:  0.75
    → Best nu: 0.05
    → Saved: Decon-COO-Results_Degradation_v4_top_10_percent_removed_nuSVR/nuSVR_CountsRMSE_Random_TSP27_random_rep4_seed9_50_1000.txt

Processing directory: Decon-COO-Resul